# 📊 INF01090 - Ciência de Dados - Regression Techniques

**House Prices - Advanced Regression Techniques - Kaggle-Style Competition**

House Prices – Advanced Regression Techniques (<https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/>) is a long-running Kaggle competition that challenges participants to predict the sale prices of homes in Ames, Iowa, using advanced regression models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 1,460 training and 1,459 test homes
- **Features:** 79 explanatory variables
- **Primary goal:** Predict house sale price


## 📂 Dataset and Objective

The dataset, compiled by Dean De Cock as an update to the classic Boston Housing dataset, describes almost every aspect of residential homes—covering lot size, room counts, materials, neighbourhood, and more. The task is to use these features to predict each property's final sale price, making it a supervised regression problem that blends statistical and machine learning techniques.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as linear regression, ridge, lasso, random forest, and gradient boosting
- **Evaluation:** models are ranked by **RMSE on log-transformed prices** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a regression pipeline that predicts **`SalePrice`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- regression modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
Id,prediction
1461,210000
1462,179500
1463,220000
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `SalePrice` should be non-negative


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline regression model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- applying `log1p(SalePrice)` during training
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [ ] `train_student.csv` loads correctly
- [ ] `test_student.csv` has the same predictor columns as expected
- [ ] your preprocessing works for both train and test
- [ ] your model produces one prediction per test row
- [ ] the output file has exactly two columns: `Id`, `prediction`
- [ ] all predictions are numeric
- [ ] all predictions are non-negative


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [35]:
import numpy as np
import pandas as pd
#import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

## 1. Load the data

Update the paths if necessary.


In [36]:
train_df = pd.read_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\train_student.csv")
test_df = pd.read_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\test_student.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (1022, 81)
Test shape: (438, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,136,20,RL,80.0,10400,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,174000
1,1453,180,RM,35.0,3675,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2006,WD,Normal,145000
2,763,60,FV,72.0,8640,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2010,Con,Normal,215200
3,933,20,RL,84.0,11670,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2007,WD,Normal,320000
4,436,60,RL,43.0,10667,Pave,NaN,IR2,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2009,ConLw,Normal,212000


## 2. Identify target and predictors


In [37]:
target_col = "SalePrice"
id_col = "Id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col].copy()

print("Target summary:")
display(y.describe())


Target summary:


count      1022.000000
mean     181312.692759
std       77617.461005
min       34900.000000
25%      130000.000000
50%      165000.000000
75%      215000.000000
max      745000.000000
Name: SalePrice, dtype: float64

### 2.1 Missing-value handling



In [ ]:
#Null values analysis
Nans = X.select_dtypes(include=["number"]).isnull().sum()
print("Numeric features with missing values:")
print(Nans[Nans > 0])
Nans = X.select_dtypes(exclude=["number"]).isnull().sum()
print("Categorical features with missing values:")
print(Nans[Nans > 0])

#Columns with more than 50% missing values are dropped
#for col in X_train.columns:
#    if(X_train[col].isnull().sum()/len(X_train)) > 0.5:
#        X_train.drop(columns=[col], inplace=True)
#        numeric_features.remove(col)
#        categorical_features.remove(col)
#    if(X_valid[col].isnull().sum()/len(X_valid)) > 0.5:
#        X_valid.drop(columns=[col], inplace=True)
#        numeric_features.remove(col)
#        categorical_features.remove(col)

#print("After dropping columns with >50% missing values:")
#print("Train shape:", X_train.shape)
#print("Valid shape:", X_valid.shape)

#See the p-values of the features (doesn't work yet)
#X_train_sm = sm.add_constant(X_train)
#model = sm.OLS(y_train, X_train_sm).fit()
#print(model.summary())

#Future ideas: drop more columns, maybe fuse some too

#No duplicated rows exist:
print("duplicated rows:", X.duplicated().sum())

#Filling missing numerical values
for col in X.select_dtypes(include=["number"]).columns:
    if(X[col].isna().sum() > 0):
        X[col] = X[col].fillna(X[col].median())

#Filling missing categorical values
for col in X.select_dtypes(exclude=["number"]).columns:
    if(X[col].isna().sum() > 0):
        X[col] = X[col].fillna("Unknown")

Numeric features with missing values:
LotFrontage    190
MasVnrArea       3
GarageYrBlt     54
dtype: int64
Categorical features with missing values:
Alley            956
MasVnrType       590
BsmtQual          26
BsmtCond          26
BsmtExposure      26
BsmtFinType1      26
BsmtFinType2      26
Electrical         1
FireplaceQu      487
GarageType        54
GarageFinish      54
GarageQual        54
GarageCond        54
PoolQC          1017
Fence            820
MiscFeature      982
dtype: int64
duplicated rows: 0


### 2.2 Checking Outliers

In [39]:
#Check outliers
quartiles = X[X.select_dtypes(include=["number"]).columns].quantile([0.01, 0.99])   
print(quartiles)

#See which values pass the quartiles
for col in X.select_dtypes(include=["number"]).columns:
    print(f"Values in {col} below the 1st percentile:")
    print(X[X[col] < quartiles.loc[0.01, col]][col])
    print(f"Values in {col} above the 99th percentile:")
    print(X[X[col] > quartiles.loc[0.99, col]][col])

           Id  MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  \
0.01    12.21        20.0        21.00   1680.00         3.00          3.0   
0.99  1444.79       190.0       143.37  44443.74         9.79          9.0   

      YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  ...  GarageArea  \
0.01    1893.68        1950.0        0.00        0.00  ...        0.00   
0.99    2009.00        2009.0      744.43     1572.79  ...      965.06   

      WoodDeckSF  OpenPorchSF  EnclosedPorch  3SsnPorch  ScreenPorch  \
0.01        0.00         0.00           0.00       0.00         0.00   
0.99      532.43       291.79         262.95     177.48       275.37   

      PoolArea  MiscVal  MoSold  YrSold  
0.01       0.0      0.0     1.0  2006.0  
0.99       0.0   1189.5    12.0  2010.0  

[2 rows x 37 columns]
Values in Id below the 1st percentile:
125     4
129     6
185     3
186     7
340    10
369     8
420     1
590    12
658     5
781     9
837     2
Name: Id, dtype: int64
Valu

### 2.3 Feature Encoding

In [40]:
#Encode some categorical variables
X["MSZoning"] = X["MSZoning"].replace({"RL": 0, "RM": 1, "C (all)": 2, "FV": 3, "RH": 4})
X["Street"] = X["Street"].replace({"Pave": 0, "Grvl": 1})
X["LotShape"] = X["LotShape"].replace({"Reg": 0, "IR1": 1, "IR2": 2, "IR3": 3})
X["LandContour"] = X["LandContour"].replace({"Lvl": 0, "Bnk": 1, "HLS": 2, "Low": 3})
X["Utilities"] = X["Utilities"].replace({"AllPub": 0, "NoSewa": 1, "NoSeWa": 2, "ELO": 3})
X["LandSlope"] = X["LandSlope"].replace({"Gtl": 0, "Mod": 1, "Sev": 2})
X["Neighborhood"] = X["Neighborhood"].replace({"CollgCr": 0, "Veenker": 1, "Crawfor": 2, "NoRidge": 3, "Mitchel": 4, "Somerst": 5, "NWAmes": 6, "OldTown": 7, "BrkSide": 8, "Sawyer": 9, "NridgHt": 10, "NAmes": 11, "SawyerW": 12, "IDOTRR": 13, "MeadowV": 14, "Edwards": 15, "Timber": 16, "Gilbert": 17, "StoneBr": 18, "ClearCr": 19, "NPknolls": 20, "Blmngtn": 21, "BrDale": 22, "SWISU": 23, "Blueste": 24})
X["Condition1"] = X["Condition1"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X["Condition2"] = X["Condition2"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X["BldgType"] = X["BldgType"].replace({"1Fam": 0, "2fmCon": 1, "Duplex": 2, "TwnhsE": 3, "Twnhs": 4})
X["HouseStyle"] = X["HouseStyle"].replace({"1Story": 0, "2Story": 1, "1.5Fin": 2, "SLvl": 3, "SFoyer": 4, "1.5Unf": 5, "2.5Unf": 6, "2.5Fin": 7})
X["CentralAir"] = X["CentralAir"].replace({"Y": 1, "N": 0})

# Convert manually encoded columns to numeric dtypes to avoid type errors
encoded_cols = ["MSZoning", "Street", "LotShape", "LandContour", "Utilities", "LandSlope", "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle", "CentralAir"]
for col in encoded_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce').astype('Int64')

#Check the number of unique values in each categorical feature
for col in X.select_dtypes(exclude=["number"]).columns:
    print(f"{col}: {X[col].nunique()} unique values")

#One-hot encode categorical variables
X_encoded = pd.get_dummies(X, columns=X.select_dtypes(exclude=["number"]).columns, drop_first=True)

Alley: 3 unique values
LotConfig: 5 unique values
RoofStyle: 6 unique values
RoofMatl: 7 unique values
Exterior1st: 14 unique values
Exterior2nd: 16 unique values
MasVnrType: 4 unique values
ExterQual: 4 unique values
ExterCond: 5 unique values
Foundation: 6 unique values
BsmtQual: 5 unique values
BsmtCond: 5 unique values
BsmtExposure: 5 unique values
BsmtFinType1: 7 unique values
BsmtFinType2: 7 unique values
Heating: 6 unique values
HeatingQC: 5 unique values
Electrical: 5 unique values
KitchenQual: 4 unique values
Functional: 7 unique values
FireplaceQu: 6 unique values
GarageType: 7 unique values
GarageFinish: 4 unique values
GarageQual: 6 unique values
GarageCond: 6 unique values
PavedDrive: 3 unique values
PoolQC: 4 unique values
Fence: 5 unique values
MiscFeature: 5 unique values
SaleType: 9 unique values
SaleCondition: 6 unique values


## 3. Build a local validation split

Use this split to compare models **before** submitting to the leaderboard.


In [41]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Transform y to log scale for training (aligns with leaderboard evaluation)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

print(X_train.shape, X_valid.shape)


(817, 80) (205, 80)


## 4. Separate numeric and categorical columns


In [42]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features)) 

Numeric features: 49
Categorical features: 31


## 5. Create a preprocessing pipeline

You may improve this pipeline as part of the assignment.


In [43]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 6. Baseline model

Start with a simple model.


In [44]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

baseline_model.fit(X_train, y_train_log)
pred_valid_baseline_log = baseline_model.predict(X_valid)
pred_valid_baseline = np.expm1(pred_valid_baseline_log)

rmse_baseline = mean_squared_error(y_valid, pred_valid_baseline) ** 0.5
mae_baseline = mean_absolute_error(y_valid, pred_valid_baseline)
r2_baseline = r2_score(y_valid, pred_valid_baseline)

print("Baseline RMSE:", rmse_baseline)
print("Baseline MAE :", mae_baseline)
print("Baseline R²  :", r2_baseline)


Baseline RMSE: 36334.46360285789
Baseline MAE : 19136.756177906544
Baseline R²  : 0.7880552016726389


## 7. Improved model

Try at least one stronger model and compare the result.


In [56]:
from sklearn.linear_model import Lasso
from sklearn.ensemble import GradientBoostingRegressor

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

"""improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Lasso(alpha=0.1))
])"""

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
    ))
])

improved_model.fit(X_train, y_train_log)
pred_valid_improved_log = improved_model.predict(X_valid)
pred_valid_improved = np.expm1(pred_valid_improved_log)

rmse_improved = mean_squared_error(y_valid, pred_valid_improved) ** 0.5
mae_improved = mean_absolute_error(y_valid, pred_valid_improved)
r2_improved = r2_score(y_valid, pred_valid_improved)

print("Improved RMSE:", rmse_improved)
print("Improved MAE :", mae_improved)
print("Improved R²  :", r2_improved)


Improved RMSE: 26864.51709059017
Improved MAE : 16450.836670497258
Improved R²  : 0.8841373833603339


## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [57]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "RMSE": [rmse_baseline, rmse_improved],
    "MAE": [mae_baseline, mae_improved],
    "R2": [r2_baseline, r2_improved],
})

comparison


,Model,RMSE,MAE,R2
0,Baseline,36334.463603,19136.756178,0.788055
1,Improved,26864.517091,16450.836670,0.884137


**Write a short discussion here.**

- Which model performed better?

The Improved model performed better, because it has the minimum errors (lower RMSE and MAE) and the maximun fit with the data (greater R2).

- Was the improvement large or small?

It was a large improvement, about 5% increase in the R2, 15% decrease with the RMSE and 20% decrease in the MAE.

- What might explain the difference?

The fact that the Improved model uses a more sophisticated and complex non-parametric method (Random Forest) while the Baseline model only uses a simpler Linear Regression.

## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [48]:
final_model = improved_model  # change if needed

y_log = np.log1p(y)
final_model.fit(X, y_log)

# Preprocess test_df the same way as X
# Fill missing values in test_df
for col in test_df.select_dtypes(include=["number"]).columns:
    if(test_df[col].isna().sum() > 0):
        test_df[col] = test_df[col].fillna(test_df[col].median())

for col in test_df.select_dtypes(exclude=["number"]).columns:
    if(test_df[col].isna().sum() > 0):
        test_df[col] = test_df[col].fillna("Unknown")

# Apply the same encodings to test_df
test_df["MSZoning"] = test_df["MSZoning"].replace({"RL": 0, "RM": 1, "C (all)": 2, "FV": 3, "RH": 4})
test_df["Street"] = test_df["Street"].replace({"Pave": 0, "Grvl": 1})
test_df["LotShape"] = test_df["LotShape"].replace({"Reg": 0, "IR1": 1, "IR2": 2, "IR3": 3})
test_df["LandContour"] = test_df["LandContour"].replace({"Lvl": 0, "Bnk": 1, "HLS": 2, "Low": 3})
test_df["Utilities"] = test_df["Utilities"].replace({"AllPub": 0, "NoSewa": 1, "NoSeWa": 2, "ELO": 3})
test_df["LandSlope"] = test_df["LandSlope"].replace({"Gtl": 0, "Mod": 1, "Sev": 2})
test_df["Neighborhood"] = test_df["Neighborhood"].replace({"CollgCr": 0, "Veenker": 1, "Crawfor": 2, "NoRidge": 3, "Mitchel": 4, "Somerst": 5, "NWAmes": 6, "OldTown": 7, "BrkSide": 8, "Sawyer": 9, "NridgHt": 10, "NAmes": 11, "SawyerW": 12, "IDOTRR": 13, "MeadowV": 14, "Edwards": 15, "Timber": 16, "Gilbert": 17, "StoneBr": 18, "ClearCr": 19, "NPknolls": 20, "Blmngtn": 21, "BrDale": 22, "SWISU": 23, "Blueste": 24})
test_df["Condition1"] = test_df["Condition1"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
test_df["Condition2"] = test_df["Condition2"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
test_df["BldgType"] = test_df["BldgType"].replace({"1Fam": 0, "2fmCon": 1, "Duplex": 2, "TwnhsE": 3, "Twnhs": 4})
test_df["HouseStyle"] = test_df["HouseStyle"].replace({"1Story": 0, "2Story": 1, "1.5Fin": 2, "SLvl": 3, "SFoyer": 4, "1.5Unf": 5, "2.5Unf": 6, "2.5Fin": 7})
test_df["CentralAir"] = test_df["CentralAir"].replace({"Y": 1, "N": 0})

# Convert to numeric
for col in encoded_cols:
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce').astype('Int64')

test_predictions_log = final_model.predict(test_df)
test_predictions = np.expm1(test_predictions_log)

submission = pd.DataFrame({
    id_col: test_df[id_col],
    "prediction": np.maximum(test_predictions, 0)  # keep predictions non-negative
})

submission.head()


,Id,prediction
0,893,139885.394162
1,1106,316197.662101
2,414,117936.272082
3,523,155512.304333
4,1037,310920.119832


## 10. Save the submission file


In [12]:
submission.to_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission:
- Best submission:
- Final submitted model:


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection
